Cell 1:  Environment Setup & Hardware Check

In [ ]:
import os
import sys
import torch
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

# 1. PROJECT PATH CONFIGURATION
PROJECT_ROOT = os.path.abspath(os.getcwd())
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
SRC_DIR = os.path.join(PROJECT_ROOT, "src")
CHECKPOINT_DIR = os.path.join(PROJECT_ROOT, "checkpoints")
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")

# Add 'src' to system path to import local modules
if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)

# Create necessary directories
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"✅ Project Root: {PROJECT_ROOT}")
print(f"📂 Data Directory: {DATA_DIR}")
print(f"💾 Checkpoint Directory: {CHECKPOINT_DIR}")
print(f"📊 Results Directory: {RESULTS_DIR}")

# 2. GPU VERIFICATION
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print(f"🚀 GPU Detected: {torch.cuda.get_device_name(0)}")
    print(f"   CUDA Version: {torch.version.cuda}")
    print(f"   Memory Allocated: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
else:
    print("⚠️ GPU not detected. Training will be slow on CPU.")

Cell 2: Import Modules & Set Hyperparameters

In [ ]:
# Cell 2: Load Datasets and Create DataLoaders
print("=" * 60)
print("CREATING DATALOADERS")
print("=" * 60)

import torch
from torch.utils.data import DataLoader
from torchvision import transforms
from src.dataset import PanNukeDataset

# Simple transforms without augmentation for now
simple_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("Loading training dataset...")
train_ds = PanNukeDataset(
    root_dir=DATA_DIR, 
    split='train', 
    transform=simple_transform,
    binary_mode=True
)

print("Loading validation dataset...")
val_ds = PanNukeDataset(
    root_dir=DATA_DIR, 
    split='validate', 
    transform=simple_transform,
    binary_mode=True
)

print(f"\n✅ Training samples: {len(train_ds)}")
print(f"✅ Validation samples: {len(val_ds)}")

# Quick test: check first few samples
print("\n🔍 Checking first 5 samples for nuclei...")
samples_with_nuclei = 0
for i in range(min(5, len(train_ds))):
    _, mask = train_ds[i]
    fg_pixels = (mask == 1).sum().item()
    if fg_pixels > 0:
        samples_with_nuclei += 1
        print(f"   Sample {i}: ✅ Has nuclei ({fg_pixels:,} pixels)")
    else:
        print(f"   Sample {i}: ❌ No nuclei")

print(f"\n   {samples_with_nuclei}/5 samples have nuclei")

# Create DataLoaders
BATCH_SIZE = 4  # Start with smaller batch size for RTX 3070

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,  # Set to 0 to avoid multiprocessing issues
    pin_memory=True if torch.cuda.is_available() else False
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True if torch.cuda.is_available() else False
)

print(f"\n✅ DataLoaders created")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Train batches: {len(train_loader)}")
print(f"   Val batches: {len(val_loader)}")

# Clear memory
import gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Cell 3: Data Exploration & Mask Verification

In [ ]:
# Minimal Exploration - No Visualization
print("=" * 60)
print("MINIMAL DATA EXPLORATION")
print("=" * 60)

import gc
import os
import torch
import numpy as np
from src.dataset import PanNukeDataset

# Load dataset
train_ds = PanNukeDataset(
    root_dir=DATA_DIR, 
    split='train', 
    transform=None,
    binary_mode=True
)

print(f"\n✅ Training dataset: {len(train_ds)} samples")

# Check first sample
img, mask = train_ds[0]
print(f"\nSample 0:")
print(f"   Image shape: {img.shape}")
print(f"   Mask shape: {mask.shape}")
print(f"   Unique mask values: {torch.unique(mask).numpy()}")
print(f"   Foreground pixels: {(mask == 1).sum().item():,} / {mask.numel():,}")

# Check memory (simple version without psutil)
print(f"\n💾 Memory info:")
if torch.cuda.is_available():
    print(f"   GPU Memory Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    print(f"   GPU Memory Cached: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

# Check a few more samples quickly
print(f"\nChecking first 20 samples...")
foreground_counts = []
for i in range(min(20, len(train_ds))):
    _, mask = train_ds[i]
    fg_count = (mask == 1).sum().item()
    foreground_counts.append(fg_count)
    if i % 5 == 0:
        print(f"   Sample {i}: {fg_count:,} foreground pixels")

if foreground_counts:
    print(f"\nForeground stats (first 20 samples):")
    print(f"   Mean: {sum(foreground_counts)/len(foreground_counts):.0f}")
    print(f"   Min: {min(foreground_counts)}")
    print(f"   Max: {max(foreground_counts)}")

# Clear memory
del img, mask
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\n✅ Done")

In [ ]:
# List all matching pairs
print("=" * 60)
print("LISTING MATCHING PAIRS")
print("=" * 60)

images_dir = os.path.join(DATA_DIR, "train", "images")
masks_dir = os.path.join(DATA_DIR, "train", "masks")

# Get all files
all_images = set([os.path.splitext(f)[0] for f in os.listdir(images_dir) if f.endswith('.png')])
all_masks = set([os.path.splitext(f)[0] for f in os.listdir(masks_dir) if f.endswith('.png')])

# Find common
common = sorted(all_images & all_masks)

print(f"Found {len(common)} matching pairs")
print(f"\nFirst 20 matching pairs:")
for i, name in enumerate(common[:20]):
    print(f"   {i+1}. {name}.png")

# Check a few random ones to see if they have nuclei
print(f"\n🔍 Checking 5 random pairs for nuclei...")
import random
import numpy as np
from PIL import Image

random_pairs = random.sample(common, min(5, len(common)))

for pair in random_pairs:
    mask_path = os.path.join(masks_dir, f"{pair}.png")
    mask = np.array(Image.open(mask_path))
    unique_vals = np.unique(mask)
    
    has_nuclei = 255 in unique_vals
    print(f"   {pair}.png: {'✅ Has nuclei' if has_nuclei else '❌ Empty mask'} - Values: {unique_vals}")

In [ ]:
# Cell 0: Environment Setup (RUN THIS FIRST)
print("=" * 60)
print("ENVIRONMENT SETUP")
print("=" * 60)

import os
import sys
import torch
import numpy as np
import gc

# 1. PROJECT PATH CONFIGURATION
PROJECT_ROOT = os.path.abspath(os.getcwd())
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
SRC_DIR = os.path.join(PROJECT_ROOT, "src")
CHECKPOINT_DIR = os.path.join(PROJECT_ROOT, "checkpoints")
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")

# Add 'src' to system path
if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)

# Create necessary directories
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"✅ Project Root: {PROJECT_ROOT}")
print(f"📂 Data Directory: {DATA_DIR}")
print(f"💾 Checkpoint Directory: {CHECKPOINT_DIR}")
print(f"📊 Results Directory: {RESULTS_DIR}")

# 2. GPU VERIFICATION
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print(f"🚀 GPU Detected: {torch.cuda.get_device_name(0)}")
    print(f"   CUDA Version: {torch.version.cuda}")
    # Clear GPU cache
    torch.cuda.empty_cache()
else:
    print("⚠️ GPU not detected. Training will be slow on CPU.")

print(f"✅ Setup complete!")
print("=" * 60)

In [ ]:
# Cell 1: Safe Dataset Verification (No Visualization)
print("=" * 60)
print("SAFE DATASET VERIFICATION (No Plots)")
print("=" * 60)

import os
import numpy as np
from PIL import Image
import gc

# Verify DATA_DIR exists
if not os.path.exists(DATA_DIR):
    print(f"❌ DATA_DIR not found: {DATA_DIR}")
    print("   Please check your data directory structure")
else:
    print(f"✅ DATA_DIR exists: {DATA_DIR}")

# Use one of the confirmed pairs
test_basename = "1-1000"
images_dir = os.path.join(DATA_DIR, "train", "images")
masks_dir = os.path.join(DATA_DIR, "train", "masks")

# Check if directories exist
if not os.path.exists(images_dir):
    print(f"❌ Images directory not found: {images_dir}")
else:
    print(f"✅ Images directory exists: {images_dir}")
    
if not os.path.exists(masks_dir):
    print(f"❌ Masks directory not found: {masks_dir}")
else:
    print(f"✅ Masks directory exists: {masks_dir}")

img_path = os.path.join(images_dir, f"{test_basename}.png")
mask_path = os.path.join(masks_dir, f"{test_basename}.png")

print(f"\nTesting: {test_basename}.png")
print(f"Image exists: {os.path.exists(img_path)}")
print(f"Mask exists: {os.path.exists(mask_path)}")

# Load and analyze without visualization
if os.path.exists(img_path) and os.path.exists(mask_path):
    try:
        # Load image and mask
        img = np.array(Image.open(img_path).convert("RGB"))
        mask_raw = np.array(Image.open(mask_path))
        
        print(f"\n✅ Successfully loaded files")
        print(f"   Image shape: {img.shape}")
        print(f"   Image dtype: {img.dtype}")
        print(f"   Image range: {img.min()} - {img.max()}")
        print(f"   Mask shape: {mask_raw.shape}")
        print(f"   Mask dtype: {mask_raw.dtype}")
        print(f"   Mask unique values: {np.unique(mask_raw)}")
        
        # Create binary mask (0/1)
        mask_binary = (mask_raw == 255).astype(np.uint8)
        
        # Calculate statistics
        total_pixels = mask_binary.size
        nucleus_pixels = mask_binary.sum()
        nucleus_percentage = (nucleus_pixels / total_pixels) * 100
        
        print(f"\n📊 Statistics:")
        print(f"   Total pixels: {total_pixels:,}")
        print(f"   Nucleus pixels: {nucleus_pixels:,}")
        print(f"   Nucleus percentage: {nucleus_percentage:.2f}%")
        
        # Save binary mask as numpy file (no visualization)
        np.save(os.path.join(RESULTS_DIR, "sample_binary_mask.npy"), mask_binary)
        print(f"   ✅ Binary mask saved to: {os.path.join(RESULTS_DIR, 'sample_binary_mask.npy')}")
        
        # Also save image array for reference
        np.save(os.path.join(RESULTS_DIR, "sample_image.npy"), img)
        print(f"   ✅ Image saved to: {os.path.join(RESULTS_DIR, 'sample_image.npy')}")
        
        # Free memory
        del img, mask_raw, mask_binary
        gc.collect()
        
        print(f"\n✅ Verification complete! Dataset is ready for training.")
        
    except Exception as e:
        print(f"❌ Error: {e}")
else:
    print(f"\n❌ Files not found! Trying to find an alternative...")
    # List available files
    if os.path.exists(images_dir):
        available_images = [f for f in os.listdir(images_dir) if f.endswith('.png')][:5]
        print(f"   Available images (first 5): {available_images}")
    if os.path.exists(masks_dir):
        available_masks = [f for f in os.listdir(masks_dir) if f.endswith('.png')][:5]
        print(f"   Available masks (first 5): {available_masks}")

print("\n" + "=" * 60)

Diagnostic Cell: Find Samples with Nuclei

In [ ]:
# Ultra-Light Test: Check just ONE sample
print("=" * 60)
print("ULTRA-LIGHT TEST: ONE SAMPLE")
print("=" * 60)

import os
import torch
import numpy as np
from PIL import Image
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Load one sample directly without the dataset class
images_dir = os.path.join(DATA_DIR, "train", "images")
masks_dir = os.path.join(DATA_DIR, "train", "masks")

# Get the first matching pair (from our earlier list)
# Use a file that we know exists from the matching pairs
test_image = "1-0.png"
test_mask = "1-0.png"

img_path = os.path.join(images_dir, test_image)
mask_path = os.path.join(masks_dir, test_mask)

if os.path.exists(img_path) and os.path.exists(mask_path):
    print(f"✅ Found matching pair: {test_image}")
    
    # Load image
    img = np.array(Image.open(img_path).convert("RGB"))
    mask = np.array(Image.open(mask_path))
    
    print(f"\nImage shape: {img.shape}")
    print(f"Mask shape: {mask.shape}")
    print(f"Mask unique values: {np.unique(mask)}")
    
    # Check for nuclei (value 255)
    if 255 in np.unique(mask):
        print(f"✅ Mask contains nuclei (value 255)")
        nucleus_pixels = (mask == 255).sum()
        total_pixels = mask.size
        print(f"   Nucleus pixels: {nucleus_pixels:,} / {total_pixels:,} ({nucleus_pixels/total_pixels*100:.2f}%)")
        
        # Convert to binary (0/1) for display
        mask_binary = (mask == 255).astype(np.uint8)
        
        # Save visualization
        fig, axes = plt.subplots(1, 3, figsize=(12, 4))
        
        axes[0].imshow(img)
        axes[0].set_title("Original Image")
        axes[0].axis('off')
        
        axes[1].imshow(mask, cmap='gray')
        axes[1].set_title("Raw Mask (0/255)")
        axes[1].axis('off')
        
        axes[2].imshow(mask_binary, cmap='gray')
        axes[2].set_title("Binary Mask (0/1)")
        axes[2].axis('off')
        
        plt.tight_layout()
        
        os.makedirs(RESULTS_DIR, exist_ok=True)
        save_path = os.path.join(RESULTS_DIR, "direct_sample.png")
        plt.savefig(save_path, dpi=100)
        plt.close()
        
        print(f"\n✅ Saved visualization to: {save_path}")
        
    else:
        print(f"⚠️ Mask does NOT contain value 255")
        print(f"   Values found: {np.unique(mask)}")
        
else:
    print(f"❌ Files not found:")
    print(f"   Image exists: {os.path.exists(img_path)}")
    print(f"   Mask exists: {os.path.exists(mask_path)}")

print("\n✅ Test complete!")

In [ ]:
# Cell 3: Quick Training Test (1 epoch to verify everything works)
print("=" * 60)
print("QUICK TRAINING TEST (1 Epoch)")
print("=" * 60)

import torch.nn as nn
from src.model import get_model
from src.train import CombinedLoss

# 1. Initialize model
print("\n🔧 Initializing model...")
model = get_model(num_classes=2, pretrain=True).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"   Total parameters: {total_params:,}")
print(f"   Trainable parameters: {trainable_params:,}")

# 2. Loss function (CrossEntropy only for now, simpler)
criterion = nn.CrossEntropyLoss(ignore_index=255)  # Ignore border pixels if any

# 3. Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# 4. Train 1 epoch (only 50 batches for quick test)
model.train()
running_loss = 0.0
num_batches = min(50, len(train_loader))

print(f"\n🏋️ Training for {num_batches} batches...")

for batch_idx, (images, masks) in enumerate(train_loader):
    if batch_idx >= num_batches:
        break
    
    images, masks = images.to(device), masks.to(device)
    
    optimizer.zero_grad()
    outputs = model(images)['out']
    loss = criterion(outputs, masks)
    loss.backward()
    optimizer.step()
    
    running_loss += loss.item()
    
    if (batch_idx + 1) % 10 == 0:
        print(f"   Batch {batch_idx+1}/{num_batches} - Loss: {loss.item():.4f}")

avg_loss = running_loss / num_batches
print(f"\n✅ Training test complete!")
print(f"   Average loss over {num_batches} batches: {avg_loss:.4f}")
print(f"   Model is learning successfully!")

# Check model predictions on a sample
model.eval()
with torch.no_grad():
    test_img, test_mask = train_ds[0]
    test_img = test_img.unsqueeze(0).to(device)
    output = model(test_img)['out']
    pred = torch.argmax(output, dim=1).cpu().squeeze()
    
    # Calculate Dice on this sample
    from src.utils import calculate_dice
    dice = calculate_dice(output.cpu(), test_mask.unsqueeze(0), num_classes=2)
    print(f"\n📊 Sample 0 prediction:")
    print(f"   Dice score: {dice:.4f}")
    print(f"   Predicted nuclei: {(pred == 1).sum().item():,} / {pred.numel():,} pixels")
    print(f"   Ground truth nuclei: {(test_mask == 1).sum().item():,} / {test_mask.numel():,} pixels")

# Clean up
del model, optimizer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\n✅ Test complete! Ready for full training.")

class weights to help with the imbalance

In [ ]:
# Cell 3.5: Improved Training Test with Class Weights
print("=" * 60)
print("IMPROVED TRAINING TEST (With Class Weights)")
print("=" * 60)

import torch.nn as nn
from src.model import get_model
from src.train import CombinedLoss

# 1. Initialize model
model = get_model(num_classes=2, pretrain=True).to(device)

# 2. Calculate class weights based on actual distribution
# From our sample: 1,577 nuclei vs 63,959 background = 1:40 ratio
# Using inverse frequency weighting
total_pixels = 65536
nucleus_pixels = 1577
background_pixels = total_pixels - nucleus_pixels

# Class weights: inverse frequency
class_weights = torch.tensor([
    total_pixels / (2 * background_pixels),  # Background weight
    total_pixels / (2 * nucleus_pixels)       # Nucleus weight (much higher)
]).to(device)

print(f"Class weights:")
print(f"   Background (class 0): {class_weights[0]:.2f}")
print(f"   Nucleus (class 1): {class_weights[1]:.2f}")

# 3. Loss function with class weights
criterion = CombinedLoss(
    num_classes=2, 
    class_weights=class_weights, 
    ignore_index=255
)

# 4. Optimizer with different learning rates
optimizer = torch.optim.AdamW([
    {'params': model.backbone.parameters(), 'lr': 1e-5},
    {'params': model.classifier.parameters(), 'lr': 1e-4},
    {'params': model.aux_classifier.parameters(), 'lr': 1e-4},
], weight_decay=1e-4)

# 5. Train for more batches (200 instead of 50)
model.train()
running_loss = 0.0
num_batches = min(200, len(train_loader))

print(f"\n🏋️ Training for {num_batches} batches...")

for batch_idx, (images, masks) in enumerate(train_loader):
    if batch_idx >= num_batches:
        break
    
    images, masks = images.to(device), masks.to(device)
    
    optimizer.zero_grad()
    outputs = model(images)['out']
    loss = criterion(outputs, masks)
    loss.backward()
    optimizer.step()
    
    running_loss += loss.item()
    
    if (batch_idx + 1) % 20 == 0:
        print(f"   Batch {batch_idx+1}/{num_batches} - Loss: {loss.item():.4f}")

avg_loss = running_loss / num_batches
print(f"\n✅ Training test complete!")
print(f"   Average loss over {num_batches} batches: {avg_loss:.4f}")

# Check predictions
model.eval()
with torch.no_grad():
    test_img, test_mask = train_ds[0]
    test_img = test_img.unsqueeze(0).to(device)
    output = model(test_img)['out']
    probs = torch.softmax(output, dim=1)
    pred = torch.argmax(output, dim=1).cpu().squeeze()
    
    # Calculate metrics
    from src.utils import calculate_dice
    dice = calculate_dice(output.cpu(), test_mask.unsqueeze(0), num_classes=2)
    
    predicted_nuclei = (pred == 1).sum().item()
    actual_nuclei = (test_mask == 1).sum().item()
    max_prob = probs.max().item()
    
    print(f"\n📊 Sample 0 prediction:")
    print(f"   Dice score: {dice:.4f}")
    print(f"   Predicted nuclei: {predicted_nuclei:,} / {pred.numel():,} pixels")
    print(f"   Ground truth nuclei: {actual_nuclei:,} / {test_mask.numel():,} pixels")
    print(f"   Max prediction probability: {max_prob:.4f}")
    
    if predicted_nuclei > 0:
        print(f"   ✅ Model is starting to detect nuclei!")
    else:
        print(f"   ⚠️ Model still predicting no nuclei - needs more training")

# Clean up
del model, optimizer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Full Training

In [ ]:
# Cell 4: Full Training with Proper Settings
print("=" * 60)
print("FULL TRAINING")
print("=" * 60)

from src.model import get_model
from src.train import CombinedLoss, train_model
from torch.utils.data import DataLoader
import json

# Hyperparameters
BATCH_SIZE = 8
EPOCHS = 30
NUM_CLASSES = 2

# Class weights (from our calculation)
class_weights = torch.tensor([0.51, 20.78]).to(device)

print(f"📊 Class weights:")
print(f"   Background (class 0): {class_weights[0]:.2f}")
print(f"   Nucleus (class 1): {class_weights[1]:.2f}")

# Recreate DataLoaders
train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True if torch.cuda.is_available() else False
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True if torch.cuda.is_available() else False
)

print(f"\n✅ DataLoaders ready")
print(f"   Train batches: {len(train_loader)}")
print(f"   Val batches: {len(val_loader)}")

# Initialize model
model = get_model(num_classes=NUM_CLASSES, pretrain=True).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n📊 Model statistics:")
print(f"   Total parameters: {total_params:,}")
print(f"   Trainable parameters: {trainable_params:,}")

# Loss function
criterion = CombinedLoss(
    num_classes=NUM_CLASSES, 
    class_weights=class_weights, 
    ignore_index=255
)

# Optimizer with different learning rates
optimizer = torch.optim.AdamW([
    {'params': model.backbone.parameters(), 'lr': 1e-5},
    {'params': model.classifier.parameters(), 'lr': 1e-4},
    {'params': model.aux_classifier.parameters(), 'lr': 1e-4},
], weight_decay=1e-4)

# Scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', patience=5, factor=0.5)

print(f"\n🚀 Starting training...")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Epochs: {EPOCHS}")
print(f"   Training samples: {len(train_ds)}")
print(f"   Validation samples: {len(val_ds)}")
print("=" * 60)

# Train
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    num_classes=NUM_CLASSES,
    epochs=EPOCHS,
    checkpoint_dir=CHECKPOINT_DIR
)

print("\n" + "=" * 60)
print("✅ Training completed!")
print(f"   Best validation Dice: {max(history['val_dice']):.4f}")
print(f"   Final validation Dice: {history['val_dice'][-1]:.4f}")
print("=" * 60)

# Save results
results = {
    'best_dice': max(history['val_dice']),
    'final_dice': history['val_dice'][-1],
    'train_loss': history['train_loss'],
    'val_loss': history['val_loss'],
    'val_dice': history['val_dice'],
    'class_weights': [0.51, 20.78],
    'batch_size': BATCH_SIZE,
    'epochs': EPOCHS
}

with open(os.path.join(RESULTS_DIR, 'training_results.json'), 'w') as f:
    json.dump(results, f, indent=2)

print(f"\n💾 Results saved to: {os.path.join(RESULTS_DIR, 'training_results.json')}")

In [ ]:
# Cell 8: Export Model for Deployment
print("=" * 60)
print("EXPORTING MODEL FOR DEPLOYMENT")
print("=" * 60)

import torch

# Load best model
best_model_path = os.path.join(CHECKPOINT_DIR, "best_model.pth")
checkpoint = torch.load(best_model_path, map_location='cpu')

# Save in different formats

# 1. Save just the model weights
torch.save(checkpoint['model_state_dict'], os.path.join(CHECKPOINT_DIR, "model_weights.pth"))
print(f"✅ Model weights saved to: {os.path.join(CHECKPOINT_DIR, 'model_weights.pth')}")

# 2. Save full checkpoint with metadata
full_checkpoint = {
    'model_state_dict': checkpoint['model_state_dict'],
    'dice': checkpoint['dice'],
    'epoch': checkpoint['epoch'],
    'config': {
        'num_classes': 2,
        'input_size': 256,
        'model_type': 'DeepLabV3_ResNet50',
        'best_dice': checkpoint['dice']
    }
}
torch.save(full_checkpoint, os.path.join(CHECKPOINT_DIR, "final_model.pth"))
print(f"✅ Full model saved to: {os.path.join(CHECKPOINT_DIR, 'final_model.pth')}")

# 3. Export to ONNX for deployment (optional)
try:
    from src.model import get_model
    
    # Create model instance
    model = get_model(num_classes=2, pretrain=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    
    # Create dummy input
    dummy_input = torch.randn(1, 3, 256, 256)
    
    # Export to ONNX
    torch.onnx.export(
        model, dummy_input,
        os.path.join(CHECKPOINT_DIR, "model.onnx"),
        input_names=['input'],
        output_names=['output'],
        dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}},
        opset_version=11
    )
    print(f"✅ ONNX model saved to: {os.path.join(CHECKPOINT_DIR, 'model.onnx')}")
except Exception as e:
    print(f"⚠️ ONNX export failed: {e}")

print("\n✅ Model export complete!")

In [ ]:
# Cell 6: Final Results Analysis (No Visualization, Just Data)
print("=" * 60)
print("FINAL RESULTS ANALYSIS")
print("=" * 60)

import json
import numpy as np
import os

# Load training history
history_path = os.path.join(RESULTS_DIR, 'training_results.json')
if os.path.exists(history_path):
    with open(history_path, 'r') as f:
        results = json.load(f)
    
    print(f"\n📊 Training Summary:")
    print(f"   Best Validation Dice: {results['best_dice']:.4f}")
    print(f"   Final Validation Dice: {results['final_dice']:.4f}")
    
    best_epoch = results['val_dice'].index(results['best_dice'])
    print(f"   Best Epoch: {best_epoch + 1}")
    
    print(f"\n📈 Loss Values:")
    print(f"   Final Train Loss: {results['train_loss'][-1]:.4f}")
    print(f"   Final Val Loss: {results['val_loss'][-1]:.4f}")
    
    print(f"\n📊 Dice Progression:")
    print(f"   First 5 epochs: {[round(d, 4) for d in results['val_dice'][:5]]}")
    print(f"   Last 5 epochs: {[round(d, 4) for d in results['val_dice'][-5:]]}")
    
    # Save summary to file (no plotting)
    summary = {
        'best_dice': results['best_dice'],
        'final_dice': results['final_dice'],
        'best_epoch': best_epoch + 1,
        'final_train_loss': results['train_loss'][-1],
        'final_val_loss': results['val_loss'][-1],
        'dice_improvement': results['val_dice'][-1] - results['val_dice'][0]
    }
    
    with open(os.path.join(RESULTS_DIR, 'summary.json'), 'w') as f:
        json.dump(summary, f, indent=2)
    
    print(f"\n✅ Summary saved to: {os.path.join(RESULTS_DIR, 'summary.json')}")

print("\n" + "=" * 60)

In [ ]:
# Cell 8: Export Model for Deployment (Safe)
print("=" * 60)
print("EXPORTING MODEL FOR DEPLOYMENT")
print("=" * 60)

import torch
import json
import os

# Load best model
best_model_path = os.path.join(CHECKPOINT_DIR, "best_model.pth")
if os.path.exists(best_model_path):
    checkpoint = torch.load(best_model_path, map_location='cpu')
    
    print(f"✅ Loaded best model")
    print(f"   Epoch: {checkpoint['epoch'] + 1}")
    print(f"   Dice: {checkpoint['dice']:.4f}")
    
    # 1. Save just the model weights
    torch.save(checkpoint['model_state_dict'], 
               os.path.join(CHECKPOINT_DIR, "model_weights.pth"))
    print(f"✅ Model weights saved: model_weights.pth")
    
    # 2. Save full checkpoint with metadata
    full_checkpoint = {
        'model_state_dict': checkpoint['model_state_dict'],
        'dice': checkpoint['dice'],
        'epoch': checkpoint['epoch'],
        'config': {
            'num_classes': 2,
            'input_size': 256,
            'model_type': 'DeepLabV3_ResNet50',
            'best_dice': checkpoint['dice']
        }
    }
    torch.save(full_checkpoint, 
               os.path.join(CHECKPOINT_DIR, "final_model.pth"))
    print(f"✅ Full model saved: final_model.pth")
    
    # 3. Save model info as JSON
    model_info = {
        'model_type': 'DeepLabV3_ResNet50',
        'num_classes': 2,
        'input_size': [256, 256],
        'best_dice': checkpoint['dice'],
        'epoch': checkpoint['epoch'] + 1,
        'class_weights': [0.51, 20.78],
        'binary_mode': True
    }
    
    with open(os.path.join(CHECKPOINT_DIR, "model_info.json"), 'w') as f:
        json.dump(model_info, f, indent=2)
    print(f"✅ Model info saved: model_info.json")
    
    print(f"\n💾 All files saved to: {CHECKPOINT_DIR}")
    
else:
    print(f"❌ Best model not found at: {best_model_path}")

print("\n✅ Export complete!")

Fixed Cell 10: Demo Notebook Setup

In [ ]:
# Cell 10: DEMO NOTEBOOK - Inference and Visualization (FIXED)
"""
DEMO: Medical Image Segmentation with DeepLabV3
Author: Your Name
Date: 2026
"""

print("=" * 70)
print("🎯 MEDICAL IMAGE SEGMENTATION DEMO")
print("   Nuclei Detection with DeepLabV3")
print("=" * 70)

import os
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import matplotlib
matplotlib.use('Agg')  # Safe backend for notebook
import warnings
warnings.filterwarnings('ignore')

# Setup paths
PROJECT_ROOT = os.path.abspath(os.getcwd())
SRC_DIR = os.path.join(PROJECT_ROOT, "src")
CHECKPOINT_DIR = os.path.join(PROJECT_ROOT, "checkpoints")
DEMO_DIR = os.path.join(PROJECT_ROOT, "demo_results")
SAMPLE_DIR = os.path.join(PROJECT_ROOT, "sample_images")

# Create directories
os.makedirs(DEMO_DIR, exist_ok=True)
os.makedirs(SAMPLE_DIR, exist_ok=True)

# Add src to path
if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)

# Import modules
from src.model import get_model
from src.utils import calculate_dice, calculate_iou

# Check device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\n📱 Device: {device}")

# Load model - FIXED: Handle state dict properly
print("\n🔧 Loading model...")
best_model_path = os.path.join(CHECKPOINT_DIR, "best_model.pth")

if os.path.exists(best_model_path):
    # Load checkpoint
    checkpoint = torch.load(best_model_path, map_location='cpu')
    
    # Create model with same configuration as training
    model = get_model(num_classes=2, pretrain=True)  # Use pretrain=True to match architecture
    
    # Load state dict with strict=False to ignore mismatched keys
    if 'model_state_dict' in checkpoint:
        state_dict = checkpoint['model_state_dict']
        best_dice = checkpoint.get('dice', 0)
        best_epoch = checkpoint.get('epoch', 0)
    else:
        state_dict = checkpoint
        best_dice = 0
        best_epoch = 0
    
    # Remove 'aux_classifier' keys if they cause issues (optional)
    # This handles cases where the saved model has aux_classifier but current doesn't
    new_state_dict = {}
    for key, value in state_dict.items():
        # Keep only necessary keys
        if 'aux_classifier' in key:
            # Skip aux_classifier if we don't need it
            continue
        new_state_dict[key] = value
    
    # Load with strict=False to ignore missing/unexpected keys
    missing_keys, unexpected_keys = model.load_state_dict(new_state_dict, strict=False)
    
    if missing_keys:
        print(f"   ⚠️ Missing keys: {missing_keys[:3]}...")
    if unexpected_keys:
        print(f"   ⚠️ Unexpected keys: {unexpected_keys[:3]}...")
    
    model.eval()
    
    if device == 'cuda':
        model = model.to(device)
    
    print(f"✅ Model loaded successfully!")
    print(f"   Best Dice: {best_dice:.4f}")
    print(f"   Best Epoch: {best_epoch + 1 if best_epoch else 'N/A'}")
else:
    print(f"❌ Model not found at {best_model_path}")
    print("   Please train the model first or check the path")
    sys.exit(1)

print("\n" + "=" * 70)

In [ ]:
# Cell: Final Summary Report
print("=" * 70)
print("FINAL TRAINING SUMMARY")
print("=" * 70)

import json
import os

# Define paths (use existing variables)
PROJECT_ROOT = os.path.abspath(os.getcwd())
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")
CHECKPOINT_DIR = os.path.join(PROJECT_ROOT, "checkpoints")

# Load training results if they exist
training_results_path = os.path.join(RESULTS_DIR, "training_results.json")

if os.path.exists(training_results_path):
    with open(training_results_path, 'r') as f:
        results = json.load(f)
    
    print(f"\n🎯 MODEL PERFORMANCE")
    print(f"   Best Validation Dice: {results['best_dice']:.4f} ({results['best_dice']*100:.2f}%)")
    print(f"   Final Validation Dice: {results['final_dice']:.4f} ({results['final_dice']*100:.2f}%)")
    
    # Calculate improvement if enough epochs
    if len(results['val_dice']) > 1:
        improvement = results['val_dice'][-1] - results['val_dice'][0]
        print(f"   Improvement: {improvement:.4f} ({improvement*100:.2f}%)")
    
    print(f"\n📊 TRAINING DETAILS")
    print(f"   Total Epochs: {len(results['train_loss'])}")
    print(f"   Final Train Loss: {results['train_loss'][-1]:.4f}")
    print(f"   Final Val Loss: {results['val_loss'][-1]:.4f}")
    
else:
    print(f"\n⚠️ Training results not found at: {training_results_path}")
    print("   Run training first to generate results")

# Check if best model exists
best_model_path = os.path.join(CHECKPOINT_DIR, "best_model.pth")
if os.path.exists(best_model_path):
    checkpoint = torch.load(best_model_path, map_location='cpu')
    best_dice = checkpoint.get('dice', 0)
    best_epoch = checkpoint.get('epoch', 0)
    
    print(f"\n💾 MODEL CHECKPOINT")
    print(f"   Location: {best_model_path}")
    print(f"   Best Dice: {best_dice:.4f}")
    print(f"   Epoch: {best_epoch + 1}")
else:
    print(f"\n⚠️ Model checkpoint not found at: {best_model_path}")

print(f"\n📁 SAVED FILES")
print(f"   Checkpoint: {CHECKPOINT_DIR}/best_model.pth")
print(f"   Training History: {RESULTS_DIR}/training_results.json")

if os.path.exists(RESULTS_DIR):
    files = [f for f in os.listdir(RESULTS_DIR) if f.endswith(('.png', '.json', '.csv'))]
    if files:
        print(f"   Results files: {len(files)} files in {RESULTS_DIR}")

print(f"\n✅ Summary complete!")
print("=" * 70)